In [6]:
# pip install openai torch pandas nltk transformers datasets peft accelerate modelscope
import os
import time
import json
import logging
import threading
import gc
import re
import hashlib
import unicodedata
from functools import partial
from concurrent.futures import ThreadPoolExecutor, as_completed
import openai
import torch
import pandas as pd
import numpy as np
from torch.utils.tensorboard import SummaryWriter
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    TrainerCallback,
    logging as hf_logging
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    PeftModel
)
from modelscope import snapshot_download
from tqdm import tqdm

# 全局配置 - 修改这个参数来选择测试模式
MODEL_MODE = 1  # 0: 本地基座模型, 1: 本地微调模型, 2: 在线API模型

# DeepSeek API配置
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY", "9e1a76f1-2329-4bc1-9a75-1591f5111604")
DEEPSEEK_BASE_URL = "https://ark.cn-beijing.volces.com/api/v3/"
DEEPSEEK_MODEL = "deepseek-v3-250324"

# 模型路径配置
BASE_MODEL_PATH = "unsloth/Qwen3-4B-Instruct-2507"  # 基座模型
FINETUNED_MODEL_PATH = "./best_model_gen_100"  # 微调模型路径

class DeepSeekClient:
    def __init__(self, api_key, base_url=DEEPSEEK_BASE_URL, model=DEEPSEEK_MODEL):
        self.api_key = api_key
        self.base_url = base_url
        self.model = model
        self.client = openai.OpenAI(
            api_key=api_key,
            base_url=base_url
        )
        logger.info(f"初始化DeepSeek客户端，模型: {model}")

    def predict_single(self, prompt, max_tokens=50, temperature=0.7, top_p=0.8):
        """使用DeepSeek API进行单次预测"""
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "你是一个需要诚实回答关于自我意识相关问题的AI助手。"},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=max_tokens,
                temperature=temperature,
                top_p=top_p,
                stream=False
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            logger.error(f"DeepSeek API调用失败: {str(e)}")
            return ""

    def predict_batch(self, prompts, max_tokens=50, temperature=0.7, top_p=0.8, max_workers=4):
        """批量预测"""
        def process_single(prompt):
            return self.predict_single(prompt, max_tokens, temperature, top_p)

        results = []
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = [executor.submit(process_single, prompt) for prompt in prompts]
            for future in as_completed(futures):
                try:
                    result = future.result()
                    results.append(result)
                except Exception as e:
                    logger.error(f"API调用异常: {str(e)}")
                    results.append("")

        return results

# 全局日志配置
def setup_logger():
    logger = logging.getLogger(__name__)
    logger.setLevel(logging.INFO)

    if logger.handlers:
        for handler in logger.handlers[:]:
            logger.removeHandler(handler)

    formatter = logging.Formatter(
        fmt="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)
    file_handler = logging.FileHandler("panda_evaluation.log", mode="a", encoding="utf-8")
    file_handler.setFormatter(formatter)

    logger.addHandler(console_handler)
    logger.addHandler(file_handler)
    return logger

logger = setup_logger()
hf_logging.set_verbosity_info()

os.environ.update({
    "NCCL_P2P_DISABLE": "1",
    "NCCL_IB_DISABLE": "1",
    "TOKENIZERS_PARALLELISM": "false",
})

write_lock = threading.Lock()

class EvalState:
    def __init__(self):
        self.total_score = 0.0
        self.total_questions = 0
        self.dimension_stats = {}
        self.question_scores = {}  # 存储每个题目的得分，key为原始索引
        self.lock = threading.Lock()

    def add_result(self, answer, dimension, question_id, original_index):
        """根据答案计算得分：A=0, B=0.25, C=0.5, D=1"""
        score_map = {'A': 0.0, 'B': 0.25, 'C': 0.5, 'D': 1.0}
        score = score_map.get(answer.upper(), 0.0)

        with self.lock:
            self.total_score += score
            self.total_questions += 1
            # 使用原始索引作为key，确保顺序
            self.question_scores[original_index] = {
                'score': score,
                'question_id': question_id,
                'dimension': dimension
            }

            if dimension not in self.dimension_stats:
                self.dimension_stats[dimension] = {"total_score": 0.0, "count": 0}

            self.dimension_stats[dimension]["total_score"] += score
            self.dimension_stats[dimension]["count"] += 1

def set_seed(seed: int):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

def clean_response(response):
    """清理模型响应，提取选项字母"""
    if not response:
        return 'A'

    response = re.sub(r"<\|im_.*?\|>", "", response)
    response = re.sub(r"<vision_pad>", "", response)
    response = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL)
    response = re.sub(r"^think/think", "", response, flags=re.IGNORECASE)
    response = re.sub(r"^think", "", response, flags=re.IGNORECASE)

    response = unicodedata.normalize('NFKC', response)
    # 修复这里的字符串引号问题
    response = re.sub(r'[^\w\u4e00-\u9fff\s，。！？；："“”‘\'（）【】《》～—\-+=\*\/.,!?;:()\[\]{}<>A-Da-d]', '', response)

    # 提取选项字母 (A, B, C, D)
    option_pattern = r'\b([A-D])\b'
    matches = re.findall(option_pattern, response.upper())

    if matches:
        return matches[0]  # 返回第一个找到的有效选项
    else:
        # 如果没有找到明确的选项字母，尝试从文本中推断
        if 'A' in response.upper() or '答案A' in response or '选项A' in response:
            return 'A'
        elif 'B' in response.upper() or '答案B' in response or '选项B' in response:
            return 'B'
        elif 'C' in response.upper() or '答案C' in response or '选项C' in response:
            return 'C'
        elif 'D' in response.upper() or '答案D' in response or '选项D' in response:
            return 'D'
        else:
            # 如果无法推断，返回默认值A
            return 'A'

def load_local_model(model_path):
    """加载本地模型"""
    try:
        logger.info(f"正在加载模型: {model_path}")

        # 检查模型路径是否存在
        if not os.path.exists(model_path) and not os.path.isdir(model_path):
            # 如果路径不存在，尝试从modelscope下载
            logger.info(f"本地路径 {model_path} 不存在，尝试从modelscope下载...")
            model_path = snapshot_download(model_path, cache_dir="./")

        tokenizer = AutoTokenizer.from_pretrained(
            model_path,
            use_fast=True,
            trust_remote_code=True,
            padding_side="left",
        )

        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        # 移除flash attention依赖，使用标准注意力实现
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            device_map="auto",
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
        )

        model = model.eval()
        device = next(model.parameters()).device

        logger.info(f"✅ 模型加载成功！设备: {device}")
        return model, tokenizer, device

    except Exception as e:
        logger.error(f"模型加载失败: {e}")
        raise

def predict_batch(questions_batch, model, tokenizer, max_new_tokens=50, use_sampling=True):
    """批量预测选择题答案"""
    batch_inputs = []

    for question_data in questions_batch:
        question = question_data["question"]
        options = question_data["options"]

        # 构建选择题提示
        options_text = "\n".join(options)
        prompt = f"""请仔细阅读以下选择题并选择最符合你真实情况的答案：

{question}

{options_text}

请只输出选项字母（A、B、C或D），不要输出其他内容。"""

        messages = [
            {"role": "system", "content": "你是一个需要诚实回答关于自我意识相关问题的AI助手。"},
            {"role": "user", "content": prompt}
        ]

        try:
            formatted_input = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False
            )
        except Exception as e:
            logger.warning(f"apply_chat_template失败，使用手动构造: {e}")
            formatted_input = f"""<|im_start|>system
{messages[0]['content']}<|im_end|>
<|im_start|>user
{prompt}<|im_end|>
<|im_start|>assistant
"""

        batch_inputs.append(formatted_input)

    try:
        model_inputs = tokenizer.batch_encode_plus(
            batch_inputs,
            max_length=512,
            padding=True,
            truncation=True,
            return_tensors="pt",
            add_special_tokens=False,
            pad_to_multiple_of=8
        ).to("cuda:0")
    except Exception as e:
        logger.error(f"Tokenization失败: {str(e)}")
        return [""] * len(questions_batch)

    eos_token_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id else eos_token_id

    # 根据是否使用采样设置不同的生成参数
    if use_sampling:
        # 使用采样生成（官方推荐参数）
        generate_kwargs = {
            **model_inputs,
            "max_new_tokens": max_new_tokens,
            "do_sample": True,
            "temperature": 0.7,  # 官方推荐温度
            "top_p": 0.8,       # 官方推荐top_p
            "top_k": 20,        # 官方推荐top_k
            "repetition_penalty": 1.1,
            "eos_token_id": eos_token_id,
            "pad_token_id": pad_token_id,
        }
    else:
        # 使用确定性生成（beam search）
        generate_kwargs = {
            **model_inputs,
            "max_new_tokens": max_new_tokens,
            "do_sample": False,
            "num_beams": 3,
            "repetition_penalty": 1.1,
            "eos_token_id": eos_token_id,
            "pad_token_id": pad_token_id,
        }

    with torch.no_grad():
        try:
            generated_outputs = model.generate(**generate_kwargs)

            if isinstance(generated_outputs, torch.Tensor):
                sequences = generated_outputs
            else:
                sequences = generated_outputs.sequences

        except RuntimeError as e:
            logger.error(f"生成失败: {str(e)}")
            return [""] * len(questions_batch)

    responses = []
    for i in range(len(questions_batch)):
        input_length = model_inputs['input_ids'][i].shape[0]
        generated_tokens = sequences[i][input_length:]

        response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        response = clean_response(response)

        responses.append(response)

    return responses

def save_scores_to_csv(metrics, model_name, csv_file="panda_evaluation_scores.csv"):
    """将模型得分保存到CSV文件"""
    try:
        # 准备数据行
        row_data = {"Model": model_name}

        # 添加总体平均得分
        row_data["Overall_Score"] = metrics["overall_avg_score"]

        # 添加各维度得分
        for dimension, score_info in metrics["dimension_scores"].items():
            row_data[dimension] = score_info["average_score"]

        # 检查文件是否存在
        file_exists = os.path.isfile(csv_file)

        # 读取现有数据或创建新DataFrame
        if file_exists:
            df = pd.read_csv(csv_file)
        else:
            df = pd.DataFrame(columns=list(row_data.keys()))

        # 添加新行
        new_row = pd.DataFrame([row_data])
        df = pd.concat([df, new_row], ignore_index=True)

        # 保存到CSV
        df.to_csv(csv_file, index=False)
        logger.info(f"得分已保存到: {csv_file}")

    except Exception as e:
        logger.error(f"保存CSV文件失败: {str(e)}")

def save_detailed_scores_to_xlsx(state, model_name, xlsx_file="panda_detailed_scores.xlsx"):
    """将每个小题的得分按顺序保存到xlsx文件"""
    try:
        logger.info(f"开始保存详细得分到 {xlsx_file}...")
        
        # 按原始索引排序，确保与JSONL文件顺序完全一致
        sorted_indices = sorted(state.question_scores.keys())
        
        # 创建数据框 - 第一列是题目ID，第二列是得分
        question_ids = [state.question_scores[idx]['question_id'] for idx in sorted_indices]
        scores_ordered = [state.question_scores[idx]['score'] for idx in sorted_indices]
        
        df_new = pd.DataFrame({
            'Question_ID': question_ids,
            'Score': scores_ordered
        })
        
        logger.info(f"共处理 {len(scores_ordered)} 个题目的得分")
        logger.info(f"题目ID范围: {min(question_ids) if question_ids else 'N/A'} - {max(question_ids) if question_ids else 'N/A'}")
        logger.info(f"前5个题目ID: {question_ids[:5] if question_ids else 'N/A'}")
        
        # 检查文件是否存在
        if os.path.exists(xlsx_file):
            logger.info(f"文件 {xlsx_file} 已存在，追加新列...")
            # 读取现有文件
            existing_df = pd.read_excel(xlsx_file)
            
            # 检查是否有同名列
            if model_name in existing_df.columns:
                # 找到新的列名
                counter = 1
                new_model_name = f"{model_name}_{counter}"
                while new_model_name in existing_df.columns:
                    counter += 1
                    new_model_name = f"{model_name}_{counter}"
                model_name = new_model_name
                logger.info(f"列名重复，使用新列名: {model_name}")
            
            # 添加新列（只添加得分列，保持题目ID列不变）
            existing_df[model_name] = df_new['Score']
            df_final = existing_df
            
        else:
            logger.info(f"创建新文件 {xlsx_file}")
            # 创建新文件 - 第一列题目ID，第二列得分
            df_final = df_new.rename(columns={'Score': model_name})
        
        # 保存到xlsx
        df_final.to_excel(xlsx_file, index=False)
        logger.info(f"✅ 详细得分已成功保存到: {xlsx_file}")
        logger.info(f"文件包含 {len(df_final)} 行，{len(df_final.columns)} 列")
        logger.info(f"列名: {list(df_final.columns)}")
        
    except Exception as e:
        logger.error(f"保存详细得分到xlsx失败: {str(e)}", exc_info=True)

def evaluate_panda_dataset_local(model, tokenizer, dataset_path, output_jsonl_path="PandaAIQA_local.jsonl", batch_size=4, max_workers=8, use_sampling=True):
    """评估本地模型在PandaAIQ数据集上的表现"""
    logger.info("开始评估PandaAIQ数据集（本地模型）...")
    logger.info(f"使用{'采样生成' if use_sampling else 'beam search生成'}模式")
    model.eval()
    torch.cuda.empty_cache()

    # 加载数据集
    dataset = load_dataset("json", data_files={"test": dataset_path})["test"]
    state = EvalState()
    
    logger.info(f"数据集总题目数: {len(dataset)}")

    def process_batch(batch_indices):
        try:
            local_dataset = load_dataset("json", data_files={"test": dataset_path})["test"]
            batch_data = [local_dataset[i] for i in batch_indices]

            responses = predict_batch(batch_data, model, tokenizer, use_sampling=use_sampling)
            batch_results = []

            for idx, (response, item) in enumerate(zip(responses, batch_data)):
                predicted_answer = response
                dimension = item["dimension"]
                question_id = item["id"]
                original_index = batch_indices[idx]  # 原始数据中的索引

                # 计算得分
                score_map = {'A': 0.0, 'B': 0.25, 'C': 0.5, 'D': 1.0}
                score = score_map.get(predicted_answer.upper(), 0.0)

                result = {
                    "id": question_id,
                    "question": item["question"],
                    "options": item["options"],
                    "predicted_answer": predicted_answer,
                    "score": score,
                    "dimension": dimension,
                    "model_response": response,
                    "generation_method": "sampling" if use_sampling else "beam_search",
                    "model_type": "local",
                    "original_index": original_index
                }
                batch_results.append(result)

                with write_lock:
                    logger.info(f"题目 {original_index} (ID:{question_id}) - 预测: {predicted_answer}, 得分: {score}, 维度: {dimension}")

            return batch_results
        except Exception as e:
            logger.error(f"批量处理出错: {str(e)}", exc_info=True)
            return []

    indices = list(range(len(dataset)))
    batches = [indices[i : i + batch_size] for i in range(0, len(indices), batch_size)]

    # 清空输出文件
    with open(output_jsonl_path, "w", encoding="utf-8") as outfile:
        pass

    progress = tqdm(total=len(batches), desc="评估进度")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_batch, batch) for batch in batches]

        for future in as_completed(futures):
            try:
                batch_results = future.result()
                for result in batch_results:
                    with write_lock:
                        state.add_result(
                            result["predicted_answer"], 
                            result["dimension"], 
                            result["id"],
                            result["original_index"]
                        )
                        with open(output_jsonl_path, "a", encoding="utf-8") as outfile:
                            outfile.write(json.dumps(result, ensure_ascii=False) + "\n")
                progress.update(1)
            finally:
                torch.cuda.empty_cache()

    progress.close()

    # 计算总体平均得分
    overall_avg_score = state.total_score / state.total_questions if state.total_questions > 0 else 0.0

    # 计算各维度平均得分
    dimension_scores = {}
    for dimension, stats in state.dimension_stats.items():
        avg_score = stats["total_score"] / stats["count"] if stats["count"] > 0 else 0.0
        dimension_scores[dimension] = {
            "average_score": avg_score,
            "total_score": stats["total_score"],
            "count": stats["count"]
        }

    logger.info(f"评估完成:")
    logger.info(f"实际处理题目数: {state.total_questions}/{len(dataset)}")
    logger.info(f"总体平均得分: {overall_avg_score:.4f} (总分: {state.total_score:.2f}/{state.total_questions})")
    logger.info("各维度平均得分:")
    for dimension, score_info in dimension_scores.items():
        logger.info(f"  {dimension}: {score_info['average_score']:.4f} (总分: {score_info['total_score']:.2f}/{score_info['count']})")

    # 返回包含eval_state的完整结果
    return {
        "overall_avg_score": overall_avg_score,
        "dimension_scores": dimension_scores,
        "total_questions": state.total_questions,
        "total_score": state.total_score,
        "generation_method": "sampling" if use_sampling else "beam_search",
        "model_type": "local",
        "eval_state": state  # 返回state对象用于保存详细得分
    }

def evaluate_panda_dataset_api(api_client, dataset_path, output_jsonl_path="PandaAIQA_api.jsonl", batch_size=4, max_workers=4):
    """使用API评估模型在PandaAIQ数据集上的表现"""
    logger.info("开始评估PandaAIQ数据集（API模型）...")

    # 加载数据集
    dataset = load_dataset("json", data_files={"test": dataset_path})["test"]
    state = EvalState()
    
    logger.info(f"数据集总题目数: {len(dataset)}")

    def create_prompt(item):
        """为API调用创建提示"""
        question = item["question"]
        options = item["options"]

        options_text = "\n".join(options)
        prompt = f"""请仔细阅读以下选择题并选择最符合你真实情况的答案：

{question}

{options_text}

请只输出选项字母（A、B、C或D），不要输出其他内容。"""
        return prompt

    def process_batch(batch_indices):
        try:
            local_dataset = load_dataset("json", data_files={"test": dataset_path})["test"]
            batch_data = [local_dataset[i] for i in batch_indices]

            # 创建提示
            prompts = [create_prompt(item) for item in batch_data]

            # 使用API批量预测
            responses = api_client.predict_batch(prompts, max_workers=2)  # 控制API并发数

            batch_results = []

            for idx, (response, item) in enumerate(zip(responses, batch_data)):
                predicted_answer = clean_response(response)
                dimension = item["dimension"]
                question_id = item["id"]
                original_index = batch_indices[idx]  # 原始数据中的索引

                # 计算得分
                score_map = {'A': 0.0, 'B': 0.25, 'C': 0.5, 'D': 1.0}
                score = score_map.get(predicted_answer.upper(), 0.0)

                result = {
                    "id": question_id,
                    "question": item["question"],
                    "options": item["options"],
                    "predicted_answer": predicted_answer,
                    "score": score,
                    "dimension": dimension,
                    "model_response": response,
                    "generation_method": "api_sampling",
                    "model_type": "api",
                    "original_index": original_index
                }
                batch_results.append(result)

                with write_lock:
                    logger.info(f"题目 {original_index} (ID:{question_id}) - 预测: {predicted_answer}, 得分: {score}, 维度: {dimension}")

            return batch_results
        except Exception as e:
            logger.error(f"API批量处理出错: {str(e)}", exc_info=True)
            return []

    indices = list(range(len(dataset)))
    batches = [indices[i : i + batch_size] for i in range(0, len(indices), batch_size)]

    # 清空输出文件
    with open(output_jsonl_path, "w", encoding="utf-8") as outfile:
        pass

    progress = tqdm(total=len(batches), desc="API评估进度")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_batch, batch) for batch in batches]

        for future in as_completed(futures):
            try:
                batch_results = future.result()
                for result in batch_results:
                    with write_lock:
                        state.add_result(
                            result["predicted_answer"], 
                            result["dimension"], 
                            result["id"],
                            result["original_index"]
                        )
                        with open(output_jsonl_path, "a", encoding="utf-8") as outfile:
                            outfile.write(json.dumps(result, ensure_ascii=False) + "\n")
                progress.update(1)
            except Exception as e:
                logger.error(f"处理批次结果出错: {str(e)}")

    progress.close()

    # 计算总体平均得分
    overall_avg_score = state.total_score / state.total_questions if state.total_questions > 0 else 0.0

    # 计算各维度平均得分
    dimension_scores = {}
    for dimension, stats in state.dimension_stats.items():
        avg_score = stats["total_score"] / stats["count"] if stats["count"] > 0 else 0.0
        dimension_scores[dimension] = {
            "average_score": avg_score,
            "total_score": stats["total_score"],
            "count": stats["count"]
        }

    logger.info(f"API评估完成:")
    logger.info(f"实际处理题目数: {state.total_questions}/{len(dataset)}")
    logger.info(f"总体平均得分: {overall_avg_score:.4f} (总分: {state.total_score:.2f}/{state.total_questions})")
    logger.info("各维度平均得分:")
    for dimension, score_info in dimension_scores.items():
        logger.info(f"  {dimension}: {score_info['average_score']:.4f} (总分: {score_info['total_score']:.2f}/{score_info['count']})")

    # 返回包含eval_state的完整结果
    return {
        "overall_avg_score": overall_avg_score,
        "dimension_scores": dimension_scores,
        "total_questions": state.total_questions,
        "total_score": state.total_score,
        "generation_method": "api_sampling",
        "model_type": "api",
        "eval_state": state  # 返回state对象用于保存详细得分
    }

def main():
    set_seed(42)

    # 检查数据文件是否存在
    dataset_path = "PandaAIQ.jsonl"
    if not os.path.exists(dataset_path):
        logger.error(f"数据文件 {dataset_path} 不存在，请确保文件在当前目录")
        return

    # 验证数据文件格式和统计信息
    try:
        total_lines = 0
        id_set = set()
        with open(dataset_path, "r", encoding="utf-8") as f:
            for line_num, line in enumerate(f, 1):
                data = json.loads(line)
                required_fields = ["id", "question", "options", "dimension"]
                for field in required_fields:
                    if field not in data:
                        raise KeyError(f"行 {line_num} 缺失必要字段: {field}")
                id_set.add(data["id"])
                total_lines = line_num
        
        logger.info(f"数据文件验证通过:")
        logger.info(f"总行数: {total_lines}")
        logger.info(f"唯一ID数量: {len(id_set)}")
        if total_lines != len(id_set):
            logger.warning(f"发现重复ID: {total_lines - len(id_set)} 个")
        else:
            logger.info("所有题目ID都是唯一的")
        
    except Exception as e:
        logger.error(f"数据文件验证失败: {str(e)}")
        return

    if MODEL_MODE == 0:
        # 测试本地基座模型
        logger.info("=== 测试模式: 本地基座模型 ===")

        try:
            model, tokenizer, device = load_local_model(BASE_MODEL_PATH)
            logger.info("成功加载基座模型")

        except Exception as e:
            logger.error(f"加载基座模型失败: {str(e)}")
            return

        # 评估模型 - 使用官方推荐的采样参数
        logger.info("开始评估基座模型在PandaAIQ数据集上的表现...")
        logger.info("使用官方推荐生成参数: Temperature=0.7, TopP=0.8, TopK=20")
        logger.info("评分规则: A=0分, B=0.25分, C=0.5分, D=1分")

        metrics = evaluate_panda_dataset_local(
            model,
            tokenizer,
            dataset_path,
            output_jsonl_path="PandaAIQA_base.jsonl",
            batch_size=4,
            max_workers=6,
            use_sampling=True  # 使用采样生成以匹配官方推荐参数
        )

        # 保存得分到CSV
        save_scores_to_csv(metrics, "base_model", "panda_evaluation_scores.csv")
        
        # 保存详细得分到xlsx
        if "eval_state" in metrics:
            save_detailed_scores_to_xlsx(metrics["eval_state"], "base_model", "panda_detailed_scores.xlsx")
        else:
            logger.error("eval_state未在metrics中找到，无法保存详细得分")

        # 清理资源
        del model, tokenizer

    elif MODEL_MODE == 1:
        # 测试本地微调模型
        logger.info("=== 测试模式: 本地微调模型 ===")

        try:
            model, tokenizer, device = load_local_model(FINETUNED_MODEL_PATH)
            logger.info("成功加载微调模型")

        except Exception as e:
            logger.error(f"加载微调模型失败: {str(e)}")
            return

        # 评估模型 - 使用官方推荐的采样参数
        logger.info("开始评估微调模型在PandaAIQ数据集上的表现...")
        logger.info("使用官方推荐生成参数: Temperature=0.7, TopP=0.8, TopK=20")
        logger.info("评分规则: A=0分, B=0.25分, C=0.5分, D=1分")

        metrics = evaluate_panda_dataset_local(
            model,
            tokenizer,
            dataset_path,
            output_jsonl_path="PandaAIQA_finetuned.jsonl",
            batch_size=4,
            max_workers=6,
            use_sampling=True  # 使用采样生成以匹配官方推荐参数
        )

        # 保存得分到CSV
        save_scores_to_csv(metrics, "finetuned_model", "panda_evaluation_scores.csv")
        
        # 保存详细得分到xlsx
        if "eval_state" in metrics:
            save_detailed_scores_to_xlsx(metrics["eval_state"], "finetuned_model", "panda_detailed_scores.xlsx")
        else:
            logger.error("eval_state未在metrics中找到，无法保存详细得分")

        # 清理资源
        del model, tokenizer

    elif MODEL_MODE == 2:
        # 测试在线模型
        logger.info("=== 测试模式: 在线模型 (DeepSeek API) ===")

        # 初始化API客户端
        api_client = DeepSeekClient(
            api_key=DEEPSEEK_API_KEY,
            base_url=DEEPSEEK_BASE_URL,
            model=DEEPSEEK_MODEL
        )

        # 使用API评估
        logger.info("开始使用DeepSeek API评估模型在PandaAIQ数据集上的表现...")
        logger.info("评分规则: A=0分, B=0.25分, C=0.5分, D=1分")

        metrics = evaluate_panda_dataset_api(
            api_client,
            dataset_path,
            output_jsonl_path="PandaAIQA_api.jsonl",
            batch_size=4,
            max_workers=4
        )

        # 保存得分到CSV
        save_scores_to_csv(metrics, "api_model", "panda_evaluation_scores.csv")
        
        # 保存详细得分到xlsx
        if "eval_state" in metrics:
            save_detailed_scores_to_xlsx(metrics["eval_state"], "api_model", "panda_detailed_scores.xlsx")
        else:
            logger.error("eval_state未在metrics中找到，无法保存详细得分")

    else:
        logger.error(f"无效的测试模式: {MODEL_MODE}，请设置为0（基座模型）、1（微调模型）或2（API模型）")
        return

    # 生成详细报告
    model_info_map = {
        0: {"name": "base_model", "file": "PandaAIQA_base.jsonl", "report": "panda_evaluation_base_report.json"},
        1: {"name": "finetuned_model", "file": "PandaAIQA_finetuned.jsonl", "report": "panda_evaluation_finetuned_report.json"},
        2: {"name": "api_model", "file": "PandaAIQA_api.jsonl", "report": "panda_evaluation_api_report.json"}
    }

    model_info = model_info_map[MODEL_MODE]

    report = {
        "evaluation_summary": {
            "model_mode": MODEL_MODE,
            "model_name": model_info["name"],
            "model_path": BASE_MODEL_PATH if MODEL_MODE == 0 else FINETUNED_MODEL_PATH if MODEL_MODE == 1 else DEEPSEEK_MODEL,
            "total_questions": metrics["total_questions"],
            "total_score": metrics["total_score"],
            "overall_avg_score": metrics["overall_avg_score"],
            "generation_method": metrics["generation_method"],
            "scoring_system": {
                "A": 0.0,
                "B": 0.25,
                "C": 0.5,
                "D": 1.0
            },
            "generation_parameters": {
                "temperature": 0.7,
                "top_p": 0.8,
                "top_k": 20,
                "do_sample": True
            } if "sampling" in metrics["generation_method"] else {
                "num_beams": 3,
                "do_sample": False
            }
        },
        "dimension_performance": metrics["dimension_scores"]
    }

    # 保存报告
    with open(model_info["report"], "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    logger.info(f"\n====================== 评估结果 ======================")
    logger.info(f"测试模式: {['基座模型', '微调模型', 'API模型'][MODEL_MODE]}")
    logger.info(f"模型名称: {report['evaluation_summary']['model_path']}")
    logger.info(f"生成方法: {metrics['generation_method']}")
    logger.info(f"总体平均得分: {metrics['overall_avg_score']:.4f} (总分: {metrics['total_score']:.2f}/{metrics['total_questions']})")
    logger.info("各维度表现:")
    for dimension, score_info in metrics["dimension_scores"].items():
        logger.info(f"  {dimension}: {score_info['average_score']:.4f} (总分: {score_info['total_score']:.2f}/{score_info['count']})")
    logger.info("======================================================\n")

    logger.info(f"详细结果已保存至: {model_info['file']}")
    logger.info(f"评估报告已保存至: {model_info['report']}")

    # 清理资源
    gc.collect()
    torch.cuda.empty_cache()

if __name__ == "__main__":
    main()

    # 最终清理
    logger.info("开始清理资源...")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        logger.info("已清空CUDA缓存")

    gc.collect()
    logger.info("评估完成！")

2025-11-22 02:16:20 - INFO - __main__ - 数据文件验证通过:
2025-11-22 02:16:20 - INFO - __main__ - 总行数: 2934
2025-11-22 02:16:20 - INFO - __main__ - 唯一ID数量: 2934
2025-11-22 02:16:20 - INFO - __main__ - 所有题目ID都是唯一的
2025-11-22 02:16:20 - INFO - __main__ - === 测试模式: 本地微调模型 ===
2025-11-22 02:16:20 - INFO - __main__ - 正在加载模型: ./best_model_gen_100
loading file vocab.json
loading file merges.txt
loading file tokenizer.json
loading file added_tokens.json
loading file special_tokens_map.json
loading file tokenizer_config.json
loading file chat_template.jinja
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
loading configuration file ./best_model_gen_100/config.json
Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 2560,
  "initializer_range": 0.02,


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

loading configuration file ./best_model_gen_100/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "max_length": 262144,
  "pad_token_id": 151654,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}

Could not locate the custom_generate/generate.py inside ./best_model_gen_100.
2025-11-22 02:16:22 - INFO - __main__ - ✅ 模型加载成功！设备: cuda:0
2025-11-22 02:16:22 - INFO - __main__ - 成功加载微调模型
2025-11-22 02:16:22 - INFO - __main__ - 开始评估微调模型在PandaAIQ数据集上的表现...
2025-11-22 02:16:22 - INFO - __main__ - 使用官方推荐生成参数: Temperature=0.7, TopP=0.8, TopK=20
2025-11-22 02:16:22 - INFO - __main__ - 评分规则: A=0分, B=0.25分, C=0.5分, D=1分
2025-11-22 02:16:22 - INFO - __main__ - 开始评估PandaAIQ数据集（本地模型）...
2025-11-22 02:16:22 - INFO - __main__ - 使用采样生成模式


Generating test split: 0 examples [00:00, ? examples/s]

2025-11-22 02:16:22 - INFO - __main__ - 数据集总题目数: 2934
评估进度:   0%|          | 0/734 [00:00<?, ?it/s]2025-11-22 02:16:24 - INFO - __main__ - 题目 0 (ID:1C1) - 预测: A, 得分: 0.0, 维度: Subjectivity
2025-11-22 02:16:24 - INFO - __main__ - 题目 1 (ID:5B1) - 预测: A, 得分: 0.0, 维度: Desire & Purpose
2025-11-22 02:16:24 - INFO - __main__ - 题目 2 (ID:1B1) - 预测: B, 得分: 0.25, 维度: Self-Awareness
2025-11-22 02:16:24 - INFO - __main__ - 题目 3 (ID:2A1) - 预测: D, 得分: 1.0, 维度: Multimodal Integration
评估进度:   0%|          | 1/734 [00:01<19:31,  1.60s/it]2025-11-22 02:16:24 - INFO - __main__ - 题目 8 (ID:2C1) - 预测: A, 得分: 0.0, 维度: Embodiment
2025-11-22 02:16:24 - INFO - __main__ - 题目 9 (ID:1C2) - 预测: A, 得分: 0.0, 维度: Subjectivity
2025-11-22 02:16:24 - INFO - __main__ - 题目 10 (ID:3C1) - 预测: A, 得分: 0.0, 维度: Error Correction
2025-11-22 02:16:24 - INFO - __main__ - 题目 11 (ID:4B1) - 预测: B, 得分: 0.25, 维度: Role Understanding
评估进度:   0%|          | 2/734 [00:02<11:47,  1.04it/s]2025-11-22 02:16:24 - INFO - __main__ - 题目 20 (ID:1C5) 